# Applied Machine Learning, Notebook 9
## The yield curve in three numbers: PCA on Chilean UF bonds

**Course:** *Applied Machine Learning* · **Prepared by:** David Díaz, with assistance from Claude (Anthropic) · **Environment:** Google Colab

### The problem

A pension fund (in Chile, an **AFP**, a private pension fund administrator), a life insurer or a bank treasury
holds Central Bank and Treasury bonds at 2, 5, 10, 20 and 30 years. The value of that portfolio depends on
**five rates at once**, one per maturity: the **yield curve** (or term structure of interest rates). To measure
its risk, hedge it or imagine scenarios, we would have to model five series that move together, with all their
correlations.

The observation that changes everything comes from Litterman and Scheinkman (1991): although the curve has many
maturities, it almost always moves in **three ways**:

- **Level**: the whole curve moves up or down together.
- **Slope**: short maturities move differently from long ones, and the curve steepens or flattens.
- **Curvature**: the middle maturities move differently from the ends, and the curve bends.

We do not need to impose these three shapes: **PCA** finds them on its own, from the data. It is the same tool
as in notebook 8, now used for what its name promises, **dimensionality reduction**: going from five rates to
three factors with economic meaning. And here, unlike in fraud detection, what matters are the first
components, not the last ones.

### What you will learn today

1. How to prepare a real financial series for PCA: gaps, frequency, **levels versus changes**, and units.
2. How to read the result: explained variance, **loadings** as curve shapes, **factors** over time.
3. How to **evaluate** the model: reconstruction error in basis points, an out-of-sample test, and stability.
4. How to use it: stress scenarios, a comparison between Chile and the United States, and going back from factors to rates.

The data are the Central Bank of Chile's curve of bonds denominated in UF, taken from its public Statistical
Database, and the US Treasury curve as a reference. The **UF** (*Unidad de Fomento*) is a Chilean unit of
account that is indexed daily to inflation.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import io, os, re, html, urllib.request, warnings

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.width", 130)

## 1. The data: the Central Bank's UF curve

**BCU** bonds (issued by the Central Bank) and **BTU** bonds (issued by the Treasury) pay in **UF**, the unit
that adjusts with inflation. Their yield is therefore a **real rate**: what the bond earns above inflation.
(Their peso-denominated counterparts are **BCP** and **BTP**, which pay a nominal rate.) Every day the Central
Bank publishes the market yield of a representative bond (the *benchmark*) at each maturity, in the table
"Interest rates on bonds auctioned by the BCCh and the Treasury" of its Statistical Database.

The next cell reads those series directly from the Central Bank's public web page, maturity by maturity, and
saves them in `curvas_chile.csv` so they do not have to be downloaded again. If the page does not respond, it
uses the course copy (`URL_DATOS`). It also brings the nominal peso curve (BCP and BTP), which we use in an exercise.

In [2]:
URL_DATOS = ""    # <-- optional: link to the course copy (curvas_chile.csv)
COPIA_CURSO = "https://raw.githubusercontent.com/daviddiazsolis/dimred_finance_playground/main/data/curvas_chile.csv"

def normalizar_link(url):
    '''Turns a Dropbox or Google Drive link into a direct download link.'''
    if "dropbox.com" in url:
        return url.replace("dl=0", "dl=1").replace("www.dropbox.com", "dl.dropboxusercontent.com")
    if "drive.google.com" in url and "/d/" in url:
        return "https://drive.google.com/uc?export=download&id=" + url.split("/d/")[1].split("/")[0]
    return url

MESES = {"Ene": 1, "Feb": 2, "Mar": 3, "Abr": 4, "May": 5, "Jun": 6, "Jul": 7, "Ago": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dic": 12}

def serie_bcch(codigo, cuadro):
    '''Reads a daily series from the public page of the Central Bank's Statistical Database.'''
    url = f"https://si3.bcentral.cl/siete/ES/Siete/Cuadro/{cuadro}?idSerie={codigo}"
    pagina = urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"}), timeout=60).read().decode("utf-8", "ignore")
    filas = re.findall(r"<tr><td>([^<]+)</td><td>([^<]*)</td></tr>", pagina)
    fechas = [pd.Timestamp(int(f.split(".")[2]), MESES[f.split(".")[1]], int(f.split(".")[0])) for f, _ in filas]
    valores = [float(v.replace(".", "").replace(",", ".")) if v.strip() else np.nan for _, v in filas]
    return pd.Series(valores, index=fechas).sort_index()

def cargar_curvas():
    if os.path.exists("curvas_chile.csv"):
        return pd.read_csv("curvas_chile.csv", index_col=0, parse_dates=True)
    if URL_DATOS:
        return pd.read_csv(normalizar_link(URL_DATOS), index_col=0, parse_dates=True)
    try:
        series = {}
        for a in [1, 2, 5, 10, 20, 30]:
            series[f"UF_{a}A"] = serie_bcch(f"F022.BUF.TIS.AN{a:02d}.UF.Z.D", "CAP_EST_OMA/MN_EST_OMA/TMS_16")
            print(f"  UF {a} years: {len(series[f'UF_{a}A'])} days")
        for a in [1, 2, 5, 10]:
            series[f"CLP_{a}A"] = serie_bcch(f"F022.BCLP.TIS.AN{a:02d}.NO.Z.D", "CAP_EST_OMA/MN_EST_OMA/TMS_15")
            print(f"  pesos {a} years: {len(series[f'CLP_{a}A'])} days")
        curvas = pd.DataFrame(series)
    except Exception as error:
        print("Could not read the Central Bank page (", error, "); using the course copy.")
        curvas = pd.read_csv(COPIA_CURSO, index_col=0, parse_dates=True)
    curvas.index.name = "fecha"
    curvas.to_csv("curvas_chile.csv")
    return curvas

curvas = cargar_curvas()
PLAZOS = ["UF_2A", "UF_5A", "UF_10A", "UF_20A", "UF_30A"]
ANIOS = [2, 5, 10, 20, 30]
print("Days:", len(curvas), " from", curvas.index.min().date(), "to", curvas.index.max().date())
curvas[PLAZOS].describe().round(2)

Days: 5985  from 2002-09-10 to 2026-09-29


,UF_2A,UF_5A,UF_10A,UF_20A,UF_30A
count,4268.00,5586.00,5586.00,4438.00,3018.00
mean,1.40,1.91,2.29,2.77,2.34
std,1.35,0.99,0.98,1.07,0.77
min,-2.92,-1.84,-0.65,0.00,0.20
25%,0.77,1.25,1.63,2.00,1.85
50%,1.54,2.24,2.41,2.78,2.35
75%,2.39,2.61,2.96,3.39,2.83
max,4.71,3.78,4.48,5.30,4.07


In [3]:
fig = go.Figure()
for fecha in ["2008-06-30", "2010-06-30", "2013-06-28", "2019-11-29", "2020-07-31", "2021-12-30"]:
    f = curvas[PLAZOS].loc[:fecha].dropna().index[-1]
    fig.add_scatter(x=ANIOS, y=curvas.loc[f, PLAZOS].values, mode="lines+markers", name=str(f.date()))
fig.update_layout(title="The UF bond curve at different moments", xaxis_title="maturity (years)",
                  yaxis_title="real rate (%)", height=420)
fig.show()

fig = px.line(curvas[PLAZOS].loc["2008":"2021"], title="The five rates over time (2008 to 2021)")
fig.update_layout(yaxis_title="real rate (%)", xaxis_title="", height=400, legend_title="maturity")
fig.show()

Two charts that already tell almost the whole story. In the first, the curve changes in **level** (in 2008 it was
around 3%; in July 2020, with the pandemic and the policy rate at its floor, short maturities ended up at
**negative** real rates), in **slope** (almost flat in 2008, very steep in 2020) and in **shape**. In the second,
the five series clearly move together, and the 2-year one is the most jittery.

## 2. Preparing the data: the part that decides the result

### 2.1 The gaps

PCA needs a complete table: each row, one day with all five rates. The real Central Bank series has gaps, and it
is worth understanding why before filling them. The official note of the table explains it: the series for each
maturity is the yield of the bond that is closest to that maturity at the time, and there are days (or years)
when **no benchmark bond exists** for that maturity. Let us look at the fraction of days with data, by year.

In [4]:
cobertura = curvas[PLAZOS].notna().groupby(curvas.index.year).mean()
fig = px.imshow(cobertura.T, color_continuous_scale="Blues", zmin=0, zmax=1, aspect="auto", text_auto=".0%",
                title="Fraction of days with data, by maturity and year")
fig.update_layout(height=330, xaxis_title="", yaxis_title="")
fig.show()

Until 2007 there is no 30-year bond; between 2008 and 2021 coverage is good at all five maturities; from 2022 on
it becomes irregular (the 2-year maturity almost disappears in 2022 and 2025, and the 20- and 30-year ones drop
to fewer than half of the days). Three decisions, each with its reason:

1. **Window: 2008 to 2021.** This is the period in which all five maturities exist. We use the recent years in
   an exercise, to see how a model that has already been estimated is applied to data with gaps.
2. **Weekly frequency.** We take the last available rate of each week (Friday). That way a maturity that did
   not trade on Friday but did on Thursday still counts, and the week is complete without inventing data.
3. **Never fill backward.** It is tempting to use `bfill()` to cover the start of a series, but that copies the
   **first future value** into all earlier days: it invents a rate that never existed and leaves it flat. Look
   at what it would do with the 30-year maturity, which starts in 2008.

In [5]:
relleno_atras = curvas["UF_30A"].loc["2006":"2008-06"].bfill()
print("Distinct values of the 30-year maturity between 2006 and March 2008 if filled backward:",
      relleno_atras.loc[:"2008-03-25"].nunique(), " (all equal to", relleno_atras.iloc[0], ")")

semanal = curvas[PLAZOS].resample("W-FRI").last()
semanal = semanal.loc["2008-01-01":"2021-12-31"].dropna()
print("Complete weeks 2008 to 2021:", len(semanal))

Distinct values of the 30-year maturity between 2006 and March 2008 if filled backward: 1  (all equal to 3.13 )
Complete weeks 2008 to 2021: 660


More than two years of a perfectly constant 30-year rate: PCA would read it as a maturity that does not move,
and all the correlations of that period would come out wrong. With weekly frequency we are left with 660
complete weeks.

### 2.2 Levels or changes

The second decision is the most important one. We can apply PCA to the **levels** of the rates (the table as it
is) or to their **changes** from one week to the next. The classic course example used levels and found that the
first component explains about 90% of the variance. Let us see why that number is misleading, with a ten-line
experiment: five **independent random walks**, that is, five series that have nothing to do with each other.

In [6]:
pc1_niveles, pc1_cambios = [], []
for semilla in range(50):
    paseos = np.cumsum(np.random.default_rng(semilla).normal(size=(660, 5)), axis=0)    # 5 unrelated series
    pc1_niveles.append(PCA().fit(StandardScaler().fit_transform(paseos)).explained_variance_ratio_[0])
    pc1_cambios.append(PCA().fit(StandardScaler().fit_transform(np.diff(paseos, axis=0))).explained_variance_ratio_[0])
print(f"Five INDEPENDENT random walks, 50 repetitions:")
print(f"  PCA on levels: component 1 explains on average {np.mean(pc1_niveles):.0%} (between {min(pc1_niveles):.0%} and {max(pc1_niveles):.0%})")
print(f"  PCA on changes: component 1 explains on average {np.mean(pc1_cambios):.0%}  (as it should: 1/5 = 20%)")

niveles_uf = PCA().fit(StandardScaler().fit_transform(semanal)).explained_variance_ratio_
print("\nUF curve, PCA on levels: component 1 =", round(niveles_uf[0], 3))

Five INDEPENDENT random walks, 50 repetitions:
  PCA on levels: component 1 explains on average 53% (between 29% and 74%)
  PCA on changes: component 1 explains on average 22%  (as it should: 1/5 = 20%)

UF curve, PCA on levels: component 1 = 0.916


Five series with **no** relationship at all, and PCA on levels says that a single component explains more than
half of their variance. This is the PCA version of the **spurious regression** of Granger and Newbold (1974):
series with trends (which do not revert to a mean, that is, are not **stationary**) look alike by chance, and PCA
mistakes that chance for structure. On changes, the same experiment gives the 20% that corresponds to five
independent series.

That is why the 90% of the curve in levels mixes two things: the real relationship between maturities and the
fact that rates have long trends. From here on we work with **weekly changes**, in **basis points** (1 basis
point = 0.01 percentage points), which is how the risk of a bond portfolio is measured: how much the rate moves
in a period.

One detail: we only compute the change between two **consecutive** weeks; if there was an incomplete week
between two complete ones, that change would span two weeks, so we discard it.

In [7]:
consecutiva = semanal.index.to_series().diff().dt.days == 7
cambios = (semanal.diff() * 100)[consecutiva].dropna()          # basis points per week
print("Weekly changes:", len(cambios), " from", cambios.index.min().date(), "to", cambios.index.max().date())
pd.DataFrame({"std. dev. (bp per week)": cambios.std(), "largest change (bp)": cambios.abs().max()}).round(1).T

Weekly changes: 620  from 2008-04-04 to 2021-12-31


,UF_2A,UF_5A,UF_10A,UF_20A,UF_30A
std. dev. (bp per week),20.2,14.4,12.5,10.0,9.9
largest change (bp),118.0,102.0,114.0,90.0,90.0


### 2.3 Units: standardize or not?

All columns are in the same unit (basis points), so we could apply PCA without standardizing, and that is the
usual practice for the US Treasury curve. But look at the standard deviations: the 2-year maturity moves 20 basis
points per week, twice as much as the 20- and 30-year ones. Without standardizing, PCA looks for the direction of
maximum variance and finds it mostly in the 2-year maturity. Let us compare the loadings of component 1 with and
without standardization.

In [8]:
sin_est = PCA().fit(cambios)
con_est = PCA().fit(StandardScaler().fit_transform(cambios))
pd.DataFrame({"not standardized": sin_est.components_[0], "standardized": con_est.components_[0]}, index=PLAZOS).round(2).T

,UF_2A,UF_5A,UF_10A,UF_20A,UF_30A
not standardized,0.67,0.49,0.40,0.29,0.26
standardized,0.32,0.47,0.49,0.48,0.46


Without standardizing, component 1 loads 0.67 on the 2-year maturity and barely 0.3 on the long ones: it is almost
"the 2-year maturity", not "the whole curve". Standardized, the five loadings are similar: a clean level. The
2-year UF maturity is especially noisy because a short inflation-indexed bond is very sensitive to inflation in
the coming months (the UF indexation it will accrue until maturity weighs heavily on its real yield), and that
source of variation is almost specific to that maturity. To understand the **shape** of curve movements we use
**standardized** changes, as in the course example; exercise 2 comes back to this decision.

## 3. PCA of the curve

The model, with the notation from notebook 8: $x_t$ is the vector of the 5 standardized changes in week $t$;
$w_1, w_2, w_3$ are the loadings (`components_`); the **factors** are the scores
$f_{j,t} = w_j^{\top} x_t$ (`transform`), and the curve is approximated with three of them:

$$ x_t \approx f_{1,t}\, w_1 + f_{2,t}\, w_2 + f_{3,t}\, w_3 $$

- $x_t$: changes in week $t$ at the five maturities, standardized (mean 0, standard deviation 1 over the window).
- $w_j$: shape $j$, a vector of 5 loadings, one per maturity.
- $f_{j,t}$: how much of shape $j$ there was in week $t$.

In words: each week, the movement of the curve is a mix of three fixed shapes, and the only thing that changes
from week to week is how much of each shape there was.

In [9]:
escalador = StandardScaler().fit(cambios)
X = escalador.transform(cambios)
pca = PCA().fit(X)
var_exp = pca.explained_variance_ratio_

# The sign of a component is arbitrary (w and -w describe the same direction). We fix it for easy reading:
# level with positive loadings, slope with the long maturity positive, curvature with the 5-year maturity positive.
signos = np.sign([pca.components_[0].sum(), pca.components_[1][-1], pca.components_[2][1]])
W = pca.components_[:3] * signos[:, None]
NOMBRES = ["level", "slope", "curvature"]

print("Explained variance:", var_exp.round(3), " | cumulative 3 components:", round(var_exp[:3].sum(), 3))
print(pd.DataFrame(W, index=NOMBRES, columns=PLAZOS).round(2))

fig = make_subplots(rows=1, cols=2, subplot_titles=("The three shapes (loadings)", "Explained variance"))
for j, nombre in enumerate(NOMBRES):
    fig.add_scatter(x=ANIOS, y=W[j], mode="lines+markers", name=f"{nombre} ({var_exp[j]:.0%})", row=1, col=1)
fig.add_bar(x=[f"PC{j}" for j in range(1, 6)], y=var_exp, name="per component", row=1, col=2, showlegend=False)
fig.add_hline(y=0, line_color="gray", row=1, col=1)
fig.update_xaxes(title_text="maturity (years)", row=1, col=1)
fig.update_layout(height=420, title="PCA of weekly changes in the UF curve, 2008 to 2021")
fig.show()

Explained variance: [0.703 0.163 0.077 0.033 0.024]  | cumulative 3 components: 0.942
           UF_2A  UF_5A  UF_10A  UF_20A  UF_30A
level       0.32   0.47    0.49    0.48    0.46
slope      -0.84  -0.21    0.09    0.32    0.38
curvature  -0.42   0.59    0.43   -0.32   -0.44


The three shapes of Litterman and Scheinkman appear on their own:

- **Level** (70% of the variance): loadings with the same sign and similar size at all five maturities, somewhat
  smaller at 2 years. The whole curve moves up or down together.
- **Slope** (16%): a large negative loading at 2 years, crossing zero around 10 years and turning positive at the
  long end. When this factor rises, the short end falls and the long end rises: the curve **steepens**; when it
  falls, the curve flattens. (We fixed the sign ourselves; what matters is the shape.)
- **Curvature** (8%): the 5-year maturity (and to some extent the 10-year) with one sign, the ends with the other.
  The curve bends in the middle.

Together they explain 94% of the weekly movements. The two remaining components (3% and 2%) are noise specific
to each maturity.

### 3.1 Are they really level, slope and curvature?

Naming a component is a hypothesis, and it gets verified. The measures the market uses are simple: **level** as
the average of the changes, **slope** as the change at 30 years minus the change at 2 years, and **curvature**
(the *butterfly*) as twice the change at 5 years minus those at 2 and 30.

In [10]:
F = pd.DataFrame(X @ W.T, index=cambios.index, columns=NOMBRES)        # factors with the fixed signs
medidas = pd.DataFrame({"average of the 5 maturities": cambios.mean(axis=1),
                        "30 years minus 2 years": cambios["UF_30A"] - cambios["UF_2A"],
                        "butterfly 2x5 minus 2 and 30": 2 * cambios["UF_5A"] - cambios["UF_2A"] - cambios["UF_30A"]})
pd.DataFrame([[np.corrcoef(F[f], medidas[m])[0, 1] for m in medidas] for f in NOMBRES],
             index=NOMBRES, columns=medidas.columns).round(2)

,average of the 5 maturities,30 years minus 2 years,butterfly 2x5 minus 2 and 30
level,0.98,-0.18,0.23
slope,-0.20,0.96,0.30
curvature,-0.03,0.13,0.87


The diagonal confirms it: the level factor has a correlation of 0.98 with the average change, the slope factor
0.96 with the 30-minus-2-year spread, and the curvature factor 0.87 with the butterfly. Off the diagonal the
correlations are low: each factor is a different thing.

### 3.2 The factors over time

Adding up the factors week by week, we see the history of the curve told in three numbers.

In [11]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=[f"{n} factor, cumulative" for n in NOMBRES])
for j, n in enumerate(NOMBRES):
    fig.add_scatter(x=F.index, y=F[n].cumsum(), name=n, row=j + 1, col=1)
for fecha in ["2008-09-15", "2019-10-18", "2020-03-16", "2021-10-08"]:      # Lehman, social unrest, pandemic, October 2021
    fig.add_vline(x=pd.Timestamp(fecha), line_dash="dot", line_color="gray")
fig.update_layout(height=620, showlegend=False, title="Cumulative level, slope and curvature (standardized units)")
fig.show()
print("The 5 weeks with the largest level movement (in absolute value):")
print(F["level"].abs().sort_values(ascending=False).head(5).round(2).to_string())

The 5 weeks with the largest level movement (in absolute value):
fecha
2021-10-08    16.59
2021-10-29     8.97
2020-06-05     7.87
2020-03-06     7.55
2021-11-12     7.18


The cumulative level reproduces the history of Chilean real rates: the drop during the financial crisis, the long
decline over the decade, the collapse of 2020 and the rebound of 2021. The five weeks with the largest level
movement all fall in 2020 and 2021: March and June 2020 with the pandemic, and October and November 2021, with
inflation accelerating, the Central Bank raising its rate sharply, and the debate over new early withdrawals from
pension funds pushing long rates up. The cumulative slope shows the cycles of flattening and steepening that
accompany monetary policy.

## 4. How good is the model? Three tests

### 4.1 Reconstruction error, in basis points

Explained variance is in standardized units, which are hard to grasp. The practical question is: if I describe
each week with $k$ factors and reconstruct the five rates, by how many basis points am I off at each maturity?
To answer it we have to make the full trip back: the PCA's `inverse_transform` (from factors to standardized
changes) and the scaler's `inverse_transform` (from standardized values to basis points).

In [12]:
def error_pb(k, X_ajuste, X_eval):
    '''RMSE in basis points of the reconstruction with k components, by maturity.'''
    m = PCA(n_components=k).fit(X_ajuste)
    reconstruido_pb = escalador.inverse_transform(m.inverse_transform(m.transform(X_eval)))
    original_pb = escalador.inverse_transform(X_eval)
    return np.sqrt(((original_pb - reconstruido_pb) ** 2).mean(axis=0))

tabla_err = pd.DataFrame({f"k = {k}": error_pb(k, X, X) for k in [1, 2, 3]}, index=PLAZOS).T
tabla_err.loc["no model (std. dev.)"] = cambios.std().values
tabla_err.round(1)

,UF_2A,UF_5A,UF_10A,UF_20A,UF_30A
k = 1,16.2,6.7,4.8,4.6,5.1
k = 2,5.3,6.2,4.7,3.5,3.8
k = 3,0.9,3.3,3.4,2.9,2.7
no model (std. dev.),20.2,14.4,12.5,10.0,9.9


With level alone ($k = 1$) the error ranges from 4.6 to 6.7 basis points from 5 years onward, but at 2 years it is 16,
almost as large as its standard deviation (20): the level does not capture that maturity. The slope fixes exactly
that: with $k = 2$ the 2-year error falls to 5. With three factors the error stays between 1 and 3.4 basis points
per week at every maturity, against typical movements of 10 to 20: the three-number model reproduces the curve
with an error of a few basis points.

### 4.2 Out of sample

So far we have fitted and measured on the same weeks. The honest test: estimate the shapes with **2008 to 2014**
and use them to describe **2015 to 2021**, a period that includes Chile's 2019 social unrest and the pandemic.

In [13]:
antes = cambios.index < "2015-01-01"
esc_a = StandardScaler().fit(cambios[antes])
despues = cambios.index >= "2015-01-01"
Xa, Xd = esc_a.transform(cambios[antes]), esc_a.transform(cambios[despues])
filas = []
for k in [1, 2, 3]:
    m = PCA(n_components=k).fit(Xa)
    resid = Xd - m.inverse_transform(m.transform(Xd))
    capturado = 1 - (resid ** 2).sum() / ((Xd - Xd.mean(axis=0)) ** 2).sum()
    rmse_pb = np.sqrt(((resid * esc_a.scale_) ** 2).mean(axis=0))
    filas.append({"k": k, "variance captured 2015 to 2021": capturado, **dict(zip(PLAZOS, rmse_pb))})
pd.DataFrame(filas).set_index("k").round(3)

,variance captured 2015 to 2021,UF_2A,UF_5A,UF_10A,UF_20A,UF_30A
k,,,,,,
1,0.778,16.403,7.247,5.572,4.881,5.653
2,0.872,9.433,5.406,5.694,4.092,3.962
3,0.932,0.551,3.417,4.368,3.769,3.089


The shapes estimated with the first seven years capture 93% of the variance of the following seven, and with
three factors the error stays between 0.6 and 4.4 basis points per week, only slightly above the in-sample error.
The model did not "memorize" its sample: the shapes are a stable property of the curve.

### 4.3 Stability of the shapes

The third test directly compares the loadings estimated in each half.

In [14]:
def formas(datos):
    m = PCA(n_components=3).fit(StandardScaler().fit_transform(datos))
    s = np.sign([m.components_[0].sum(), m.components_[1][-1], m.components_[2][1]])
    return m.components_ * s[:, None]

Wa, Wd = formas(cambios[antes]), formas(cambios[despues])
print("Correlation between the 2008 to 2014 loadings and the 2015 to 2021 loadings:")
for j, n in enumerate(NOMBRES):
    print(f"  {n:10s}: {np.corrcoef(Wa[j], Wd[j])[0, 1]:.3f}")
fig = go.Figure()
for j, n in enumerate(NOMBRES):
    fig.add_scatter(x=ANIOS, y=Wa[j], mode="lines+markers", name=f"{n} 2008 to 2014", line=dict(color=px.colors.qualitative.Plotly[j]))
    fig.add_scatter(x=ANIOS, y=Wd[j], mode="lines+markers", name=f"{n} 2015 to 2021", line=dict(color=px.colors.qualitative.Plotly[j], dash="dash"))
fig.update_layout(title="The shapes in two different periods", xaxis_title="maturity (years)", yaxis_title="loading", height=420)
fig.show()

Correlation between the 2008 to 2014 loadings and the 2015 to 2021 loadings:
  level     : 0.959
  slope     : 0.958
  curvature : 0.912


Correlations of 0.96, 0.96 and 0.91: the three shapes are almost identical in the two periods. Curvature is the
least stable, as is usual: it explains little variance and its estimate is noisier.

## 5. Using the model

### 5.1 Stress scenarios

A direct use of the loadings: describe a **typical** movement of each factor in basis points. A one-standard-deviation
shock to factor $j$ moves the (standardized) curve by $\sqrt{\lambda_j}\, w_j$, where $\lambda_j$ is its
eigenvalue (`explained_variance_`); multiplying by the standard deviation of each maturity brings us back to basis
points. This is the basis of the scenarios used by regulators and risk departments: instead of inventing curve
movements, they use the ones history shows.

In [15]:
lam = pca.explained_variance_[:3]
escenarios = pd.DataFrame({n: 2 * np.sqrt(lam[j]) * W[j] * escalador.scale_ for j, n in enumerate(NOMBRES)}, index=PLAZOS)
print("Weekly movement of 2 standard deviations of each factor, in basis points:")
print(escenarios.round(1).T.to_string())
hoy = semanal.iloc[-1]
fig = go.Figure()
fig.add_scatter(x=ANIOS, y=hoy.values, mode="lines+markers", name=f"curve on {semanal.index[-1].date()}", line=dict(color="black", width=3))
for n in NOMBRES:
    fig.add_scatter(x=ANIOS, y=hoy.values + escenarios[n].values / 100, mode="lines+markers", name=f"{n} shock")
fig.update_layout(title="Stress scenarios: the last curve plus a weekly 2-standard-deviation shock to each factor",
                  xaxis_title="maturity (years)", yaxis_title="real rate (%)", height=420)
fig.show()

Weekly movement of 2 standard deviations of each factor, in basis points:


           UF_2A  UF_5A  UF_10A  UF_20A  UF_30A
level       24.0   25.4    23.2    17.8    17.0
slope      -30.6   -5.5     2.0     5.7     6.8
curvature  -10.5   10.5     6.6    -4.0    -5.4


A two-standard-deviation level movement raises the whole curve by between 17 and 25 basis points in one week
(somewhat more between 2 and 10 years than at the long end). The slope moves the short and long ends in opposite
directions, and the curvature bends the 5-year segment against the ends. A bond portfolio can be evaluated against
these three scenarios instead of against five separate rates.

### 5.2 Chile versus the United States

Are the three shapes a law of fixed income or something Chilean? We repeat exactly the same process (weekly,
changes in basis points, standardized, 2008 to 2021, maturities from 2 to 30 years) with the US Treasury curve,
published by the Department of the Treasury.

In [16]:
def curva_tesoro(desde=2008, hasta=2021):
    partes = []
    for anio in range(desde, hasta + 1):
        archivo = f"tesoro_{anio}.csv"
        if not os.path.exists(archivo):
            url = ("https://home.treasury.gov/resource-center/data-chart-center/interest-rates/daily-treasury-rates.csv/"
                   f"{anio}/all?type=daily_treasury_yield_curve&field_tdr_date_value={anio}&page&_format=csv")
            with open(archivo, "wb") as f:
                f.write(urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"}), timeout=60).read())
        partes.append(pd.read_csv(archivo))
    t = pd.concat(partes)
    t["Date"] = pd.to_datetime(t["Date"], format="%m/%d/%Y")
    return t.set_index("Date").sort_index()[["2 Yr", "5 Yr", "10 Yr", "20 Yr", "30 Yr"]]

tesoro = curva_tesoro()
sem_us = tesoro.resample("W-FRI").last().loc["2008-01-01":"2021-12-31"].dropna()
cambios_us = (sem_us.diff() * 100)[sem_us.index.to_series().diff().dt.days == 7].dropna()
pca_us = PCA().fit(StandardScaler().fit_transform(cambios_us))
s_us = np.sign([pca_us.components_[0].sum(), pca_us.components_[1][-1], pca_us.components_[2][1]])
W_us = pca_us.components_[:3] * s_us[:, None]

print("Weeks:", len(cambios_us))
print(pd.DataFrame({"Chile UF": var_exp[:5], "US": pca_us.explained_variance_ratio_[:5]}, index=[f"PC{j}" for j in range(1, 6)]).round(3).T)
print("\nWeekly standard deviation (bp):")
print(pd.DataFrame({"Chile UF": cambios.std().values, "US": cambios_us.std().values}, index=ANIOS).round(1).T)

fig = make_subplots(rows=1, cols=3, subplot_titles=NOMBRES)
for j in range(3):
    fig.add_scatter(x=ANIOS, y=W[j], mode="lines+markers", name="Chile UF", line=dict(color="#2563eb"), showlegend=j == 0, row=1, col=j + 1)
    fig.add_scatter(x=ANIOS, y=W_us[j], mode="lines+markers", name="US", line=dict(color="#dc2626"), showlegend=j == 0, row=1, col=j + 1)
fig.update_layout(height=360, title="The three shapes in Chile and in the United States")
fig.show()

Weeks: 730
            PC1    PC2    PC3    PC4    PC5
Chile UF  0.703  0.163  0.077  0.033  0.024
US        0.843  0.130  0.021  0.004  0.002

Weekly standard deviation (bp):
            2     5     10    20    30
Chile UF  20.2  14.4  12.5  10.0   9.9
US         8.2  11.2  11.7  11.6  11.5


The shapes are almost the same in both countries: this is a regularity of yield curves, not a Chilean accident.
What changes is **how much** they explain. In the United States the three factors explain 99.4% and the level
alone 84%; in Chile, 94% and 70%. The Chilean curve has more maturity-specific movement, especially at 2 years,
which in the United States is the least volatile maturity (8 basis points per week) and in Chile the most volatile
(20). Two likely reasons, best treated as hypotheses: the Chilean market is less deep (a benchmark that switches
bonds or trades little adds its own noise), and short UF bonds carry the effect of short-term inflation, which does
not exist in a nominal Treasury bond.

## 6. Key takeaways

1. The yield curve has many maturities but moves in **three shapes**: level, slope and curvature. PCA finds them
   without being told, and in Chile they explain 94% of weekly movements.
2. Preparation decides the result: **do not invent data** (no `bfill`), choose the window and frequency according
   to the gaps, and work with **changes**, not levels: with levels, PCA finds structure even among independent series.
3. If one maturity is much more volatile than the rest, **standardizing** prevents it from taking over the first component.
4. Factor names must be **verified** (correlation with the average, the spread and the butterfly), and the sign of
   a component is a convention.
5. The model is evaluated in business units (**basis points**), **out of sample**, and by the **stability** of its
   shapes. With three factors, the error is a few basis points per week and the shapes do not change between periods.
6. The loadings are useful in practice: stress scenarios, hedging, and comparisons across markets.

## Exercises

1. **Inflation compensation.** The dataset also includes the nominal peso curve (`CLP_2A`, `CLP_5A`, `CLP_10A`).
   The difference between the nominal and the real rate at the same maturity is **inflation compensation**
   (the *breakeven* rate): what the market expects for inflation plus a premium. Build a weekly 2008 to 2021 table
   with the changes in the six rates (2, 5 and 10 years in pesos and in UF), standardize it and apply PCA. Does a
   component appear that moves peso and UF rates in opposite directions? How much variance does it explain? Compare
   its scores with the change in 5-year inflation compensation.
2. **Standardize or not, out of sample.** Repeat the test from section 4.2 (fit on 2008 to 2014, measure on 2015 to
   2021) **without standardizing** the changes, and compare the captured variance and the error in basis points by
   maturity with the standardized version. Which of the two versions better describes the long maturities, which
   are the ones that weigh most in the portfolio of an AFP or an insurer?
3. **The model and the gaps.** Since 2022 maturities are missing in many weeks. Take the weeks from 2023 to 2026
   where all five maturities are present, compute their consecutive changes with the scaler and the PCA estimated
   on 2008 to 2021, and measure the reconstruction error with three factors in basis points. Do the 2008 to 2021
   shapes still describe the recent curve?

## Solutions

In [17]:
# Exercise 1: inflation compensation
seis = ["CLP_2A", "CLP_5A", "CLP_10A", "UF_2A", "UF_5A", "UF_10A"]
s6 = curvas[seis].resample("W-FRI").last().loc["2008-01-01":"2021-12-31"].dropna()
c6 = (s6.diff() * 100)[s6.index.to_series().diff().dt.days == 7].dropna()
p6 = PCA().fit(StandardScaler().fit_transform(c6))
print("Weeks:", len(c6), "| explained variance:", p6.explained_variance_ratio_.round(3))
print(pd.DataFrame(p6.components_[:4], index=[f"PC{j}" for j in range(1, 5)], columns=seis).round(2))
f6 = p6.transform(StandardScaler().fit_transform(c6))
bei5 = c6["CLP_5A"] - c6["UF_5A"]
print("\nCorrelation of each component with the change in 5-year inflation compensation:")
print({f"PC{j + 1}": round(np.corrcoef(f6[:, j], bei5)[0, 1], 2) for j in range(4)})

Weeks: 663 | explained variance: [0.584 0.199 0.113 0.055 0.03  0.019]
     CLP_2A  CLP_5A  CLP_10A  UF_2A  UF_5A  UF_10A
PC1    0.29    0.45     0.47   0.25   0.45    0.48
PC2   -0.49   -0.32    -0.25   0.66   0.36    0.16
PC3    0.75   -0.11    -0.26   0.53  -0.12   -0.25
PC4   -0.34    0.53     0.28   0.44  -0.42   -0.40

Correlation of each component with the change in 5-year inflation compensation:
{'PC1': np.float64(0.06), 'PC2': np.float64(-0.77), 'PC3': np.float64(0.0), 'PC4': np.float64(0.57)}


The first component (58%) is the common level: the six rates move together. The second (20%) is the direction we
were looking for: negative loadings on the three peso rates and positive on the three UF rates. It is the
**expected inflation** factor: when the market expects more inflation, nominal rates rise and real rates fall, and
the gap between them (inflation compensation) widens. Its correlation with the change in 5-year compensation is
minus 0.77 (the sign, once again, is a convention). A portfolio manager can thus separate how much of the
portfolio's risk is **real rate** risk and how much is **inflation** risk, and hedge them separately.

In [18]:
# Exercise 2: standardize or not, out of sample
filas = []
for version in ["standardized", "not standardized"]:
    if version == "standardized":
        A_, D_, escala = Xa, Xd, esc_a.scale_
    else:
        A_ = cambios[antes].values - cambios[antes].values.mean(axis=0)
        D_ = cambios[despues].values - cambios[antes].values.mean(axis=0)
        escala = np.ones(5)
    m = PCA(n_components=3).fit(A_)
    r = D_ - m.inverse_transform(m.transform(D_))
    r_pb = r * escala
    d_pb = (D_ - D_.mean(axis=0)) * escala
    filas.append({"version": version, "variance captured (bp²)": 1 - (r_pb ** 2).sum() / (d_pb ** 2).sum(),
                  **dict(zip(PLAZOS, np.sqrt((r_pb ** 2).mean(axis=0))))})
pd.DataFrame(filas).set_index("version").round(3)

,variance captured (bp²),UF_2A,UF_5A,UF_10A,UF_20A,UF_30A
version,,,,,,
standardized,0.952,0.551,3.417,4.368,3.769,3.089
not standardized,0.948,0.101,1.602,4.834,3.780,4.382


Measured in basis points, both versions capture a similar fraction of total risk, but they distribute it
differently. Without standardizing, PCA focuses on reproducing the 2-year maturity (the most volatile) almost
without error and pays for it with larger errors at the long maturities. Standardized, the error is spread more
evenly and the long maturities are better described. For an AFP or an insurer, whose portfolio is concentrated in
long maturities, the standardized version is the one that better describes its risk. There is no universal answer:
the choice depends on which maturities matter to whoever uses the model.

In [19]:
# Exercise 3: the model and the gaps (2023 to 2026)
reciente = curvas[PLAZOS].resample("W-FRI").last().loc["2023-01-01":].dropna()
c_rec = (reciente.diff() * 100)[reciente.index.to_series().diff().dt.days == 7].dropna()
print("Complete, consecutive weeks since 2023:", len(c_rec))
X_rec = escalador.transform(c_rec)                        # scaler and PCA estimated on 2008 to 2021
m3 = PCA(n_components=3).fit(X)
r_rec = escalador.inverse_transform(X_rec) - escalador.inverse_transform(m3.inverse_transform(m3.transform(X_rec)))
pd.DataFrame({"recent error (bp)": np.sqrt((r_rec ** 2).mean(axis=0)),
              "2008 to 2021 error (bp)": error_pb(3, X, X),
              "recent std. dev. (bp)": c_rec.std().values}, index=PLAZOS).round(1).T

Complete, consecutive weeks since 2023: 25


,UF_2A,UF_5A,UF_10A,UF_20A,UF_30A
recent error (bp),1.1,3.9,4.3,3.5,2.9
2008 to 2021 error (bp),0.9,3.3,3.4,2.9,2.7
recent std. dev. (bp),13.9,11.3,12.5,10.4,11.3


With few complete weeks the measurement is noisy, but the message holds: the shapes estimated on 2008 to 2021
still reconstruct recent changes with errors of a few basis points, of the same order as in sample. The shapes of
the curve change slowly; what changes from one period to another is how much of each shape there is. With data
that have gaps, a more complete alternative is to estimate each week's factors using only the available maturities
(a regression of the observed changes on the loadings of those maturities), which is the basis of factor models for
incomplete curves.